# Counts

In [20]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
from collections import defaultdict
from typing import List, Dict, Tuple, Set, Any
import os
from pathlib import Path

json_file = 'MY_ROOT_FOLDER/research/continual_segmentation/standard_framework/datasets/ADEChallengeData2016/ade20k_panoptic_train.json'


random.seed(42)
np.random.seed(42)
plot_colors = ['blue', 'green', 'red', 'orange', 'purple',   'black']


with open(json_file, "r") as f:
    original_data = json.load(f)
    
def image_counts():
    has_only_base_imgs =set()
    has_only_new_imgs =set()
    has_base_and_new=set()
    has_new=set()
    has_base=set()
    has_no_new=set()
    has_none=set()
    all_imgs = set()
    
    img_ann = dict()
    
    
        
    for ann in original_data['annotations']:
        img_id =  ann['image_id']
        ann_segms = ann['segments_info']
        all_imgs.add(img_id)
        img_ann[img_id] = ann
        
        # if not ann_segms:
        #     print(f'{img_id} has no annotations')
            
        for s_info in ann_segms:
            cat_id = s_info['category_id']
            
            if cat_id < 100:
                has_base.add(img_id)
            elif cat_id <= 149:
                has_new.add(img_id)

            
            
    has_base_and_new   = has_base.intersection(has_new) 
    has_only_base_imgs = has_base.difference(has_new)    
    has_only_new_imgs = has_new.difference(has_base)     
    has_no_new = all_imgs.difference(has_new)
    has_none =all_imgs.difference(has_new).difference(has_base)
    
    set_distrib = dict(
        all_imgs=(sorted(list(all_imgs)), len(all_imgs)),
        has_new=(sorted(list(has_new)), len(has_new)),
        has_base=(sorted(list(has_base)), len(has_base)),
        has_base_and_new=(sorted(list(has_base_and_new)), len(has_base_and_new)),
        has_only_base_imgs=(sorted(list(has_only_base_imgs)), len(has_only_base_imgs)),
        has_only_new_imgs=(sorted(list(has_only_new_imgs)), len(has_only_new_imgs)),
        has_no_new=(sorted(list(has_no_new)), len(has_no_new)),
        has_none=(sorted(list(has_none)), len(has_none))
    )
    
        
    # print(f"\nHave none of the 150 classes: {has_none}")   
     
    return dict(set_distrib=set_distrib, img_ann=img_ann   )     
            
imgid_to_ann = image_counts().get('img_ann')
set_distrib = image_counts().get('set_distrib')

for k,v in set_distrib.items():
    print(k,v[1])

all_imgs 20210
has_new 9390
has_base 20193
has_base_and_new 9386
has_only_base_imgs 10807
has_only_new_imgs 4
has_no_new 10820
has_none 13


# Category Representation

In [21]:
def print_plot_class_distributions(img_lists_dict, num_classes=150, normalized=False,plot_title =None):
    """
    Plots normalized class distributions for each image list in img_lists_dict.
    
    Args:
        img_id_to_ann: dict mapping img_id to ADE20K annotation (must have 'segments_info' with 'category_id')
        img_lists_dict: dict mapping list name to list of img_ids
        num_classes: total number of classes (default: 150)
    """
    pass
    dist_dict = {}
    for list_name, img_ids in img_lists_dict.copy().items():
        
        class_counts = np.zeros(num_classes, dtype=np.float32)
        total = 0
        for img_id in img_ids:
            ann = imgid_to_ann.get(img_id)
            if ann is not None and 'segments_info' in ann:
                for seg in ann['segments_info']:
                    cat_id = seg['category_id']
                    if 0 <= cat_id < num_classes:
                        class_counts[cat_id] += 1
                        total += 1
        # Normalize
        if total > 0:
            class_dist = class_counts / total if normalized else class_counts
        else:
            class_dist = class_counts  # all zeros
        dist_dict[list_name] = class_dist

    # Create DataFrame: rows = list names, columns = class indices
    dff = pd.DataFrame.from_dict(dist_dict, orient='index', columns=[f"cls_{i}" for i in range(num_classes)])
    
    norm_word= "Normalized " if normalized else ''
    plot_title = f"{norm_word}Class Distribution per Image List" if not plot_title else plot_title
    print(f"{norm_word}Class Distribution Table:")
    print(dff.round(4).to_string())
    # Plot
    plt.figure(figsize=(6, 3), dpi=500)
    random.seed(42)
    colors = random.sample(plot_colors, len(plot_colors))
    for idx, (list_name, row) in enumerate(dff.iterrows()):
        plt.plot(range(num_classes), row.values, label=list_name, color=colors[idx])
    plt.xlabel("Class Index")
    plt.ylabel(f"{norm_word}frequency")
    plt.title(plot_title)
    plt.legend()
    plt.tight_layout()
    plt.show()
    

# print_plot_class_distributions(
#     {
#         f"{k}({v[1]})":v[0] for k,v in set_distrib.items() if k  in ['has_base', 'has_base_and_new','has_no_new'] 
#     }
# )


    

In [22]:


# print_plot_class_distributions(
#     {
#          f"{k}({v[1]})":v[0] for k,v in set_distrib.items() if k not in ['has_base', 'has_base_and_new','has_no_new','has_only_new_imgs', 'has_none']
#     }
# )

# print_plot_class_distributions(
#     {
#          f"{k}({v[1]})":v[0]  for k,v in set_distrib.items() if k  in ['has_only_new_imgs', 'has_none']
#     }
# )

In [23]:
def balanced_split_by_new_classes(images, annotations, new_class_ids, num_classes=150, random_seed=42, plot_title="Class Distribs on Images"):
    """
    Splits images into two sets so that the distribution of new classes is as similar as possible in both.
    
    Args:
        images: list of image ids or objects
        annotations: list of annotation dicts (must have 'segments_info' with 'category_id')
        new_class_ids: set or list of class ids considered "new"
        num_classes: total number of classes (default: 150)
        random_seed: for reproducibility
        
    Returns:
        (A_imgs, A_anns), (B_imgs, B_anns)
        Also plots the normalized new-class distributions for both splits.
    """
    # Build a list of (img_idx, set of new class ids in image)
    img_newclass_sets = []
    for idx, ann in enumerate(annotations):
        new_cats = set(seg['category_id'] for seg in ann['segments_info'] if seg['category_id'] in new_class_ids)
        img_newclass_sets.append((idx, new_cats))
    
    # Shuffle for randomness
    indices = list(range(len(images)))
    random.seed(42)
    random.shuffle(indices)
    print(f"Indices: {indices}")
    # Greedy assignment: alternate images to A and B, but try to keep running new-class histograms close
    A_idx, B_idx = [], []
    A_hist = np.zeros(num_classes, dtype=int)
    B_hist = np.zeros(num_classes, dtype=int)
    
    for i in indices:
        _, new_cats = img_newclass_sets[i]
        # For each image, compute the effect of adding it to A or B
        A_score = np.sum(A_hist[list(new_cats)]) if new_cats else 0
        B_score = np.sum(B_hist[list(new_cats)]) if new_cats else 0
        # Assign to the split with lower current sum for these new classes
        if A_score <= B_score:
            A_idx.append(i)
            for c in new_cats:
                A_hist[c] += 1
        else:
            B_idx.append(i)
            for c in new_cats:
                B_hist[c] += 1
    
    # Build splits
    A_imgs = [images[i] for i in A_idx]
    A_anns = [annotations[i] for i in A_idx]
    B_imgs = [images[i] for i in B_idx]
    B_anns = [annotations[i] for i in B_idx]
    
    # Compute normalized new-class distributions
    def get_newclass_dist(anns):
        counts = np.zeros(num_classes, dtype=int)
        total = 0
        for ann in anns:
            for seg in ann['segments_info']:
                cid = seg['category_id']
                if cid in new_class_ids:
                    counts[cid] += 1
                    total += 1
        return counts / total if total > 0 else counts
    
    dist_A = get_newclass_dist(A_anns)
    dist_B = get_newclass_dist(B_anns)
    
   
    # print_plot_class_distributions({f"{k} ({len(v)})":v for k,v in dict(Split_A= A_imgs, Split_B=B_imgs).items()}, plot_title =plot_title)
    A= sorted(A_imgs)
    B= sorted(B_imgs)
    return (A, get_anns(A)), (B, get_anns(B))

def get_anns(imgs):
    return [imgid_to_ann[img] for img in imgs]

has_base_and_new = set_distrib.get('has_base_and_new')
all_imgs =  set_distrib.get('all_imgs')


new_class_ids = list(range(100,150))
class_ids = list(range(0,150))


all_imgs_with_count, title  = (all_imgs,'Class Distribs on `All Images`')

imgs_only_all = sorted(list(all_imgs_with_count[0]))

print(imgs_only_all[:5])



all_splitA,all_splitB = balanced_split_by_new_classes(imgs_only_all, list(get_anns(imgs_only_all)), class_ids, plot_title =title, )

print(f"overlapping images: {set(all_splitA[0]).intersection(set(all_splitB[0]))} ")


['ADE_train_00000001', 'ADE_train_00000002', 'ADE_train_00000003', 'ADE_train_00000004', 'ADE_train_00000005']
Indices: [703, 14188, 6645, 19261, 8332, 14088, 7064, 17182, 6846, 13641, 160, 8454, 19489, 17146, 14035, 7591, 13832, 5144, 2506, 4679, 18517, 14598, 10581, 3379, 18068, 17058, 9994, 9758, 911, 16667, 9, 17860, 11439, 5162, 1958, 5643, 10760, 6417, 7138, 7507, 6924, 5267, 6190, 14593, 14678, 8843, 6446, 10588, 13666, 7068, 16025, 835, 12135, 1183, 13203, 7881, 3664, 19457, 12222, 18490, 16956, 12658, 6396, 19242, 9483, 10175, 8452, 16387, 11549, 1432, 5448, 4103, 15068, 5040, 4272, 12400, 3034, 14485, 12634, 624, 6848, 3043, 16807, 3001, 5030, 4850, 3177, 19053, 6092, 16329, 9286, 18345, 10479, 5791, 7601, 6888, 18428, 3923, 14917, 19149, 12430, 18855, 9600, 14672, 20163, 14191, 9848, 19042, 675, 4431, 4935, 3055, 12702, 14667, 17506, 12289, 5344, 13415, 8844, 14826, 20172, 18772, 4434, 8018, 19615, 5768, 610, 4149, 15351, 13870, 6383, 13740, 18530, 10818, 10283, 18944, 19171

In [24]:
all_splitA_imgs, all_splitA_ann = all_splitA
all_splitB_imgs, all_splitB_ann = all_splitB


for im in [all_splitA_imgs, all_splitB_imgs]:
    print(len(im))

print(all_splitA_imgs[:5])
print(all_splitB_imgs[:5])

10193
10017
['ADE_train_00000002', 'ADE_train_00000005', 'ADE_train_00000007', 'ADE_train_00000008', 'ADE_train_00000009']
['ADE_train_00000001', 'ADE_train_00000003', 'ADE_train_00000004', 'ADE_train_00000006', 'ADE_train_00000012']


In [25]:
def measure_overlap(images,
                    annotations,
                    base_classes,
                    new_classes,
                    threshold_k=20) -> float:
    """
    Measure overlap using the equation φ(O_B×C, k) = (1/M) * Σ[SUMROW(O(j,:)) > k]
    
    Args:
        cooccurrence_matrix: Co-occurrence matrix O_B×C
        threshold_k: Threshold for 'common' co-occurrence (uses config default if None)
        
    Returns:
        Overlap metric φ in range [0, 1] 
    """
    def compute_cooccurrence_matrix( 
                                  images: List[Dict],
                                  annotations: List[Dict], 
                                  base_classes: Set[int],
                                  new_classes: Set[int]) -> Tuple[np.ndarray, List[int], List[int]]:
        """
        Compute co-occurrence matrix O_B×C between base and new classes.
        
        Args:
            images: List of image info dictionaries
            annotations: List of annotation dictionaries
            base_classes: Set of base class IDs
            new_classes: Set of new class IDs
            
        Returns:
            Tuple of (cooccurrence_matrix, base_class_list, new_class_list)
        """
        
        # Create ordered lists for consistent indexing
        base_list = sorted(list(base_classes))
        new_list = sorted(list(new_classes))
        
        # Initialize co-occurrence matrix
        cooccurrence_matrix = np.zeros((len(base_list), len(new_list)), dtype=int)
        
        # Create mapping from class ID to matrix index
        base_to_idx = {cls_id: idx for idx, cls_id in enumerate(base_list)}
        new_to_idx = {cls_id: idx for idx, cls_id in enumerate(new_list)}
        
        # Count co-occurrences at image level (not instance level)
        for annotation in annotations:
            # Get all classes present in this image
            image_classes = set()
            for segment in annotation['segments_info']:
                image_classes.add(segment['category_id'])
            
            # Find base and new classes in this image
            image_base_classes = image_classes.intersection(base_classes)
            image_new_classes = image_classes.intersection(new_classes)
            
            # Update co-occurrence counts
            for base_cls in image_base_classes:
                for new_cls in image_new_classes:
                    base_idx = base_to_idx[base_cls]
                    new_idx = new_to_idx[new_cls]
                    cooccurrence_matrix[base_idx, new_idx] += 1
        
        return cooccurrence_matrix, base_list, new_list
   
    
    cooccurrence_matrix = compute_cooccurrence_matrix(images,annotations,base_classes,new_classes)[0]
    if threshold_k is None:
        threshold_k = self.config.threshold_k
        
    M = cooccurrence_matrix.shape[0]  # Number of base classes (B,)
    
    # Calculate row sums (SUMROW operation)
    row_sums = np.sum(cooccurrence_matrix, axis=1) # (B,)
    
    # print(f"Row sums stats: max={np.max(row_sums)}, min={np.min(row_sums)}, mean={np.mean(row_sums):.2f}")
    # print(f"Classes above threshold {threshold_k}: {np.sum(row_sums > threshold_k)} out of {len(row_sums)}")
    
    # Count how many base classes have row sum > k
    classes_above_threshold = np.sum(row_sums > threshold_k) # Z (num of base classes that meet threshold)
    
    # Calculate overlap metric
    overlap_phi = classes_above_threshold / M if M > 0 else 0.0 # |R (% of base classes that meet threshold)
    
    # print(f"Calculated η = {classes_above_threshold}/{len(row_sums)} = {overlap_phi:.4f}")
    
    return overlap_phi 


   

In [26]:
# def select_candidates_for_class_overlap(candidates, current_imgs, current_anns, 
#                                        base_imgs, base_anns, base_classes, step_classes,
#                                        needed, measure_overlap, target_class_overlap_ratio=None,
#                                        mode='add'):
#     """
#     Selects candidates (to add or remove) to achieve target class overlap with base.
    
#     Args:
#         candidates: List of (img, ann) tuples to choose from
#         current_imgs: Current images in the step
#         current_anns: Current annotations in the step
#         base_imgs: Base step images
#         base_anns: Base step annotations
#         base_classes: Set of base class IDs
#         step_classes: Set of step class IDs
#         needed: Number of candidates to select
#         measure_overlap: Function to measure class overlap
#         target_class_overlap_ratio: Target class overlap ratio (0-1), None = no targeting
#         mode: 'add' or 'remove'
        
#     Returns:
#         List of selected (img, ann) tuples
#     """
#     if target_class_overlap_ratio is None or not candidates:
#         # No class overlap targeting, return first N candidates
#         return candidates[:needed]
    
#     # Current class overlap
#     current_overlap = measure_overlap(current_imgs, current_anns, base_classes, step_classes)
    
#     print(f"    Current class overlap: {current_overlap:.2%}, Target: {target_class_overlap_ratio:.2%}")
    
#     # Score each candidate by how it affects class overlap
#     scored_candidates = []
#     for img, ann in candidates:
#         if mode == 'add':
#             # Simulate adding this image
#             test_imgs = current_imgs + [img]
#             test_anns = current_anns + [ann]
#         else:  # mode == 'remove'
#             # Simulate removing this image
#             test_imgs = [i for i in current_imgs if i != img]
#             test_anns = [a for a in current_anns if a != ann]
        
#         # Measure new overlap
#         new_overlap = measure_overlap(test_imgs, test_anns, base_classes, step_classes)
        
#         # Score: negative distance to target (closer = higher score)
#         score = -abs(new_overlap - target_class_overlap_ratio)
#         scored_candidates.append((score, img, ann))
    
#     # Sort by score (best first)
#     scored_candidates.sort(reverse=True, key=lambda x: x[0])
    
#     # Greedily select candidates that move us closer to target
#     selected = []
#     temp_imgs = list(current_imgs)
#     temp_anns = list(current_anns)
    
#     for score, img, ann in scored_candidates:
#         if len(selected) >= needed:
#             break
        
#         if mode == 'add':
#             temp_imgs.append(img)
#             temp_anns.append(ann)
#         else:  # remove
#             if img in temp_imgs:
#                 idx = temp_imgs.index(img)
#                 temp_imgs.pop(idx)
#                 temp_anns.pop(idx)
        
#         selected.append((img, ann))
    
#     # Measure final overlap
#     final_overlap = measure_overlap(temp_imgs, temp_anns, base_classes, step_classes)
#     print(f"    After selection: {final_overlap:.2%} class overlap")
    
#     return selected


In [27]:
def select_images_dual_objective_iterative(candidates, current_imgs, current_anns,
                                          base_imgs, base_anns, base_classes, step_classes,
                                          target_image_count, measure_overlap, 
                                          target_class_overlap_ratio, base_img_set,
                                          mode='add', max_iterations=100):
    """
    Iteratively selects images to simultaneously achieve both:
    1. Target image overlap count with base
    2. Target class overlap with base
    
    Args:
        candidates: List of (img, ann) tuples available for selection
        current_imgs: Current images in step
        current_anns: Current annotations in step
        base_imgs: Base step images
        base_anns: Base step annotations
        base_classes: Set of base class IDs
        step_classes: Set of step class IDs
        target_image_count: Target number of overlapping/non-overlapping images needed
        measure_overlap: Function to measure class overlap
        target_class_overlap_ratio: Target class overlap ratio (0-1)
        base_img_set: Set of base image IDs for fast lookup
        mode: 'add' or 'remove'
        max_iterations: Maximum optimization iterations
        
    Returns:
        List of selected (img, ann) tuples that best satisfy both objectives
    """
    if not candidates or target_class_overlap_ratio is None:
        return candidates[:target_image_count]
    
    #overlap_score = measure_overlap(step_imgs, step_anns, base_classes, step_classes)
    current_class_overlap = measure_overlap(current_imgs, current_anns, base_classes, step_classes)
    
    print(f"    Starting class overlap: {current_class_overlap:.2%}, Target: {target_class_overlap_ratio:.2%}")
    print(f"    Need to select: {target_image_count} images")
    
    # Multi-objective scoring function
    def compute_score(test_imgs, test_anns):
        """Score = weighted combination of both objectives"""
        # Objective 1: Class overlap distance (primary)
        #overlap_score = measure_overlap(step_imgs, step_anns, base_classes, step_classes)
        new_class_overlap = measure_overlap(test_imgs, test_anns, base_classes, step_classes)
        class_distance = abs(new_class_overlap - target_class_overlap_ratio)
        
        # Objective 2: Image count satisfaction (secondary)
        current_overlap_imgs = len(set(test_imgs) & base_img_set)
        image_count_distance = abs(len(test_imgs) - len(current_imgs))
        
        # Combined score (minimize both distances)
        # Weight class overlap more heavily (0.7 vs 0.3)
        score = -0.7 * class_distance - 0.3 * (image_count_distance / len(current_imgs))
        return score, new_class_overlap
    
    # Initialize with greedy selection
    selected = []
    remaining_candidates = list(candidates)
    temp_imgs = list(current_imgs)
    temp_anns = list(current_anns)
    
    best_score = float('-inf')
    best_class_overlap = current_class_overlap
    
    # Phase 1: Greedy initialization
    for iteration in range(min(target_image_count, len(remaining_candidates))):
        best_candidate = None
        best_iter_score = float('-inf')
        best_iter_overlap = 0
        
        for idx, (img, ann) in enumerate(remaining_candidates):
            # Simulate adding/removing this candidate
            if mode == 'add':
                test_imgs = temp_imgs + [img]
                test_anns = temp_anns + [ann]
            else:  # remove
                test_imgs = [i for i in temp_imgs if i != img]
                test_anns = [a for a in temp_anns if a != ann]
            
            score, class_overlap = compute_score(test_imgs, test_anns)
            
            if score > best_iter_score:
                best_iter_score = score
                best_iter_overlap = class_overlap
                best_candidate = (idx, img, ann)
        
        if best_candidate is None:
            break
        
        idx, img, ann = best_candidate
        selected.append((img, ann))
        remaining_candidates.pop(idx)
        
        if mode == 'add':
            temp_imgs.append(img)
            temp_anns.append(ann)
        else:
            if img in temp_imgs:
                remove_idx = temp_imgs.index(img)
                temp_imgs.pop(remove_idx)
                temp_anns.pop(remove_idx)
        
        best_score = best_iter_score
        best_class_overlap = best_iter_overlap
    
    # Phase 2: Local refinement via swapping
    improved = True
    refinement_iterations = 0
    
    while improved and refinement_iterations < max_iterations:
        improved = False
        refinement_iterations += 1
        
        # Try swapping each selected image with remaining candidates
        for i, (sel_img, sel_ann) in enumerate(selected):
            current_score, current_overlap = compute_score(temp_imgs, temp_anns)
            
            for cand_img, cand_ann in remaining_candidates[:20]:  # Check top 20 candidates
                # Simulate swap
                if mode == 'add':
                    swap_imgs = [img if img != sel_img else cand_img for img in temp_imgs]
                    swap_anns = [ann if ann != sel_ann else cand_ann for ann in temp_anns]
                else:
                    # For remove mode, swap in the removal list
                    swap_imgs = temp_imgs  # No actual swap in remove mode
                    swap_anns = temp_anns
                
                swap_score, swap_overlap = compute_score(swap_imgs, swap_anns)
                
                if swap_score > current_score + 0.001:  # Small improvement threshold
                    # Accept swap
                    selected[i] = (cand_img, cand_ann)
                    remaining_candidates.remove((cand_img, cand_ann))
                    remaining_candidates.append((sel_img, sel_ann))
                    
                    if mode == 'add':
                        idx = temp_imgs.index(sel_img)
                        temp_imgs[idx] = cand_img
                        temp_anns[idx] = cand_ann
                    
                    best_score = swap_score
                    best_class_overlap = swap_overlap
                    improved = True
                    break
            
            if improved:
                break
    
    final_class_overlap = measure_overlap(temp_imgs, temp_anns, base_classes, step_classes)
    
    print(f"    After {refinement_iterations} refinement iterations:")
    print(f"    Selected {len(selected)} images")
    print(f"    Final class overlap: {final_class_overlap:.2%} (Δ: {abs(final_class_overlap - target_class_overlap_ratio):.2%})")
    
    return selected


In [28]:
def compute_image_overlap_with_base(step_imgs, base_imgs):
    """
    Computes the overlap in images between a step and the base.
    
    Args:
        step_imgs: List of image IDs for the step
        base_imgs: List of image IDs for the base step
        
    Returns:
        float: Overlap ratio (0.0 to 1.0)
        int: Number of overlapping images
    """
    step_set = set(step_imgs)
    base_set = set(base_imgs)
    
    overlap_imgs = step_set & base_set
    overlap_count = len(overlap_imgs)
    overlap_ratio = overlap_count / len(step_set) if len(step_set) > 0 else 0.0
    
    return overlap_ratio, overlap_count



In [29]:

def create_initial_setup(split_A_imgs, split_A_anns, base_cls, inc_cls, 
                        measure_overlap, num_classes=150):
    """
    Creates initial continual learning setup using split A with maximum image overlap.
    
    Args:
        split_A_imgs: List of image IDs from split A
        split_A_anns: List of annotations from split A
        base_cls: Number of base classes (e.g., 100)
        inc_cls: Number of incremental classes per step (e.g., 5)
        measure_overlap: Function to measure class overlap (imgs, anns, base_cls, new_cls)
        num_classes: Total number of classes (default: 150)
        
    Returns:
        pd.DataFrame with columns: [step, images, imgs_overlap_base, avail_classes, 
                                   overlap_classes_base, step_classes]
        dict: {step: (imgs, anns)} for each step
    """
    # Define class order (assuming 0-149)
    class_order = list(range(num_classes))
    
    # Calculate number of steps
    remaining_classes = num_classes - base_cls
    num_inc_steps = (remaining_classes + inc_cls - 1) // inc_cls
    total_steps = 1 + num_inc_steps
    
    # Step 1: Create base step
    base_classes = set(class_order[:base_cls])
    base_imgs, base_anns = [], []
    
    for img, ann in zip(split_A_imgs, split_A_anns):
        img_classes = {seg['category_id'] for seg in ann['segments_info']}
        # Include images that have base classes
        if img_classes & base_classes:
            base_imgs.append(img)
            base_anns.append(ann)
    
    print(f"Base step: {len(base_imgs)} images with classes {list(base_classes)[:5]}...")
    
    # Store step data
    step_data = {1: (base_imgs, base_anns)}
    table_rows = []
    
    # Get available classes in base
    base_avail_classes = set()
    for ann in base_anns:
        base_avail_classes.update(seg['category_id'] for seg in ann['segments_info'])
    
    # Add base step to table
    table_rows.append({
        'step': 1,
        'num_imgs': len(base_imgs),
        'imgs_ov_base': len(base_imgs),  # 100% overlap with itself
        'img_ov_%': round(len(base_imgs)/len(base_imgs),2),
        'avail_cls': len(base_avail_classes),
        'ov_cls_base': 0.0,  # No incremental classes yet
        'step_cls': len(base_classes),
        
    })
    
    # Create incremental steps (maximize image overlap with base)
    for step in range(2, total_steps + 1):
        start_idx = base_cls + (step - 2) * inc_cls
        end_idx = min(start_idx + inc_cls, num_classes)
        step_classes = set(class_order[start_idx:end_idx])
        
        # Get all images from split A that contain any step classes
        step_imgs, step_anns = [], []
        for img, ann in zip(split_A_imgs, split_A_anns):
            img_classes = {seg['category_id'] for seg in ann['segments_info']}
            if img_classes & step_classes:
                step_imgs.append(img)
                step_anns.append(ann)
        
        # Measure overlaps
        imgs_overlap_base = len(set(step_imgs) & set(base_imgs))
        
        # Available classes in this step
        step_avail_classes = set()
        for ann in step_anns:
            step_avail_classes.update(seg['category_id'] for seg in ann['segments_info'])
        
        # Class overlap with base using provided function
        all_seen_classes = base_classes | step_classes
        overlap_score = measure_overlap(step_imgs, step_anns, base_classes, step_classes)
        
        step_data[step] = (step_imgs, step_anns)
        
        table_rows.append({
            'step': step,
            'num_imgs': len(step_imgs),
            'imgs_ov_base': imgs_overlap_base,
            'img_ov_%': round(imgs_overlap_base/len(step_imgs),2),
            'avail_cls': len(step_avail_classes),
            'ov_cls_base': overlap_score,
            'step_cls': len(step_classes)
        })
        
        print(f"Step {step}: {len(step_imgs)} images, {imgs_overlap_base} overlap with base, "
              f"class overlap: {overlap_score:.2%}")
        
        
        # Add Total row
    incremental_rows = [row for row in table_rows if row['step'] != 1]
    
    tt_row_num_imgs = sum(row['num_imgs'] for row in table_rows)
    tt_row_imgs_ov_base = np.mean([row['imgs_ov_base'] for row in incremental_rows]) if incremental_rows else 0
    tt_row_img_ov = np.mean([row['img_ov_%'] for row in incremental_rows]) if incremental_rows else 0
    
    table_rows.append({
        'step': 'Total',
        'num_imgs': tt_row_num_imgs,
        'imgs_ov_base': f"μ: {tt_row_imgs_ov_base:.2f}",
        'img_ov_%': f"μ: {tt_row_img_ov:.2f}",
        'avail_cls': f"μ: {np.mean([row['avail_cls'] for row in incremental_rows]):.2f}",
        'ov_cls_base': f"μ: {np.mean([row['ov_cls_base'] for row in incremental_rows]) if incremental_rows else 0:.2f}",
        'step_cls': '-'
    })
    
    
    df = pd.DataFrame(table_rows)
    return df, step_data



In [30]:
def reorder_candidates_by_target(candidates, target_class_overlap_ratio):
    """
    Reorder candidates to favor target class overlap distribution.
    
    Strategy:
    - target = 0.0: Lowest co-occurrence first (reverse order)
    - target = 1.0: Highest co-occurrence first (original order)
    - target = 0.5: Middle co-occurrence first
    - target = 0.75: 3rd quartile first
    
    Args:
        candidates: List of (img, ann, score) tuples sorted descending by score
        target_class_overlap_ratio: Target overlap ratio (0.0 to 1.0)
        
    Returns:
        Reordered list prioritizing candidates that match target distribution (same length as input)
    """
    if not candidates:
        return candidates
    
    n = len(candidates)
    
    # Find the index that corresponds to the target percentile
    # target=1.0 -> start at index 0 (highest)
    # target=0.5 -> start at index n//2 (middle)
    # target=0.0 -> start at index n-1 (lowest)
    target_start_idx = int((1.0 - target_class_overlap_ratio) * (n - 1))
    
    # Create circular reordering starting from target index
    # This puts candidates near the target percentile first
    reordered = []
    visited = set()
    
    # Start from target index and spiral outward
    left = target_start_idx
    right = target_start_idx + 1
    
    # Add center element first
    if left < n and left >= 0:  # *** ADDED: Bounds check ***
        reordered.append(candidates[left])
        visited.add(left)
    
    # Alternate between left and right, moving outward
    while len(reordered) < n:
        added_this_iteration = False  # *** NEW: Track if we added anything ***
        
        # Add from right
        if right < n and right not in visited:
            reordered.append(candidates[right])
            visited.add(right)
            added_this_iteration = True  # *** NEW ***
        right += 1  # *** MOVED: Always increment ***
        
        # Add from left  
        left -= 1  # *** MOVED: Decrement before check ***
        if left >= 0 and left not in visited:
            reordered.append(candidates[left])
            visited.add(left)
            added_this_iteration = True  # *** NEW ***
        
        # *** FIXED: Better termination condition ***
        if not added_this_iteration:
            # No elements added this iteration - we're done
            break
    
    # *** NEW: Safety check and fill any missing elements ***
    if len(reordered) < n:
        # Find any indices we missed
        for i in range(n):
            if i not in visited:
                reordered.append(candidates[i])
                visited.add(i)
    
    # *** FINAL CHECK: Ensure exact length match ***
    if len(reordered) != n:
        raise ValueError(f"Reordering failed: expected {n} candidates, got {len(reordered)}")
    
    return reordered


In [31]:
def run_dual_optimization(
    initial_imgs=None,  initial_anns=None,
    split_B_imgs=None, split_B_dict=None,
    base_imgs=None, base_anns=None,
    base_classes=None, step_classes=None,
    target_img_overlap_ratio=None, target_class_overlap_ratio=None,
    measure_overlap=None, max_iterations=100):
    """
    Dual optimization to achieve both image and class overlap targets simultaneously.
    
    Args:
        initial_imgs: List of initial image IDs for the step
        initial_anns: List of initial annotations for the step
        split_B_imgs: List of image IDs from split B (reserve pool)
        split_B_dict: Dict mapping img_id to annotation from split B
        base_imgs: List of base step image IDs
        base_anns: List of base step annotations
        base_classes: Set of base class IDs
        step_classes: Set of step class IDs
        target_img_overlap_ratio: Target ratio for image overlap with base (0-1)
        target_class_overlap_ratio: Target ratio for class overlap with base (0-1)
        measure_overlap: Function to measure class overlap (imgs, anns, base_cls, new_cls)
        max_iterations: Maximum number of optimization iterations
        
    Returns:
        Tuple of (best_imgs, best_anns) achieving both targets
    """
    
    # Validate inputs
    if any(x is None for x in [initial_imgs, initial_anns, split_B_imgs, split_B_dict,
                                base_imgs, base_classes, step_classes, 
                                target_img_overlap_ratio, target_class_overlap_ratio,
                                measure_overlap]):
        raise ValueError("Missing required parameters")
    
    # Create base image set for fast lookup
    base_img_set = set(base_imgs)
    
    # Calculate target image count
    target_img_count = int(len(initial_imgs) * target_img_overlap_ratio)
    
    print(f"  Dual Optimization:")
    print(f"    Target image overlap: {target_img_count} / {len(initial_imgs)} ({target_img_overlap_ratio:.1%})")
    print(f"    Target class overlap: {target_class_overlap_ratio:.1%}")

    # Start with initial split
    current_imgs = list(initial_imgs)
    current_anns = list(initial_anns)
    
    # Build candidate pools from split B
    overlapping_candidates = []
    nonoverlapping_candidates = []
    
    for img in split_B_imgs:
        if img not in initial_imgs:
            ann = split_B_dict[img]
            img_classes = {seg['category_id'] for seg in ann['segments_info']}
            if img_classes & step_classes:
                if img in base_img_set:
                    overlapping_candidates.append((img, ann))
                else:
                    nonoverlapping_candidates.append((img, ann))
    
    print(f"    Available candidates: {len(overlapping_candidates)} overlapping, {len(nonoverlapping_candidates)} non-overlapping")
    
    # Iterative optimization
    best_imgs, best_anns = current_imgs[:], current_anns[:]
    best_score = float('inf')
    
    for iteration in range(max_iterations):
        current_img_overlap = len(set(current_imgs) & base_img_set)
        current_class_overlap = measure_overlap(current_imgs, current_anns, 
                                                base_classes, step_classes)
        
        # Calculate distances from targets
        img_distance = abs(current_img_overlap - target_img_count)
        class_distance = abs(current_class_overlap - target_class_overlap_ratio)
        combined_score = img_distance + class_distance * 100  # Weight class overlap higher
        
        if combined_score < best_score:
            best_score = combined_score
            best_imgs, best_anns = current_imgs[:], current_anns[:]
        
        # Check convergence
        if img_distance == 0 and class_distance < 0.01:
            print(f"    ✓ Converged at iteration {iteration}")
            break
        
        # Decide action
        if current_img_overlap < target_img_count:
            # Add overlapping image that improves class overlap
            if overlapping_candidates:
                # Score candidates
                best_candidate = None
                best_candidate_score = float('inf')
                
                for img, ann in overlapping_candidates[:10]:  # Check top 10
                    test_imgs = current_imgs + [img]
                    test_anns = current_anns + [ann]
                    test_class_overlap = measure_overlap(test_imgs, test_anns,
                                                        base_classes, step_classes)
                    score = abs(test_class_overlap - target_class_overlap_ratio)
                    
                    if score < best_candidate_score:
                        best_candidate_score = score
                        best_candidate = (img, ann)
                
                if best_candidate:
                    current_imgs.append(best_candidate[0])
                    current_anns.append(best_candidate[1])
                    overlapping_candidates.remove(best_candidate)
            else:
                break
                
        elif current_img_overlap > target_img_count:
            # Remove overlapping image that maintains class overlap
            overlap_imgs = [(img, ann) for img, ann in zip(current_imgs, current_anns) 
                            if img in base_img_set]
            
            if overlap_imgs:
                best_removal = None
                best_removal_score = float('inf')
                
                for img, ann in overlap_imgs:
                    test_imgs = [i for i in current_imgs if i != img]
                    test_anns = [a for i, a in zip(current_imgs, current_anns) if i != img]
                    test_class_overlap = measure_overlap(test_imgs, test_anns,
                                                        base_classes, step_classes)
                    score = abs(test_class_overlap - target_class_overlap_ratio)
                    
                    if score < best_removal_score:
                        best_removal_score = score
                        best_removal = (img, ann)
                
                if best_removal:
                    idx = current_imgs.index(best_removal[0])
                    current_imgs.pop(idx)
                    current_anns.pop(idx)
                    
                    # Add replacement from non-overlapping
                    if nonoverlapping_candidates:
                        replacement = nonoverlapping_candidates.pop(0)
                        current_imgs.append(replacement[0])
                        current_anns.append(replacement[1])
            else:
                break
        else:
            # Fine-tune class overlap only
            if abs(current_class_overlap - target_class_overlap_ratio) < 0.01:
                break
            
            # Try swapping to adjust class overlap
            if current_class_overlap < target_class_overlap_ratio:
                # Need to increase class overlap - swap in images with more base class co-occurrence
                if overlapping_candidates or nonoverlapping_candidates:
                    # Find best swap
                    candidates = overlapping_candidates + nonoverlapping_candidates
                    best_swap = None
                    best_swap_score = float('inf')
                    
                    for new_img, new_ann in candidates[:5]:
                        for i, (old_img, old_ann) in enumerate(zip(current_imgs, current_anns)):
                            test_imgs = current_imgs[:i] + [new_img] + current_imgs[i+1:]
                            test_anns = current_anns[:i] + [new_ann] + current_anns[i+1:]
                            test_class_overlap = measure_overlap(test_imgs, test_anns,
                                                                base_classes, step_classes)
                            score = abs(test_class_overlap - target_class_overlap_ratio)
                            
                            if score < best_swap_score:
                                best_swap_score = score
                                best_swap = (i, new_img, new_ann, old_img in base_img_set, new_img in base_img_set)
                    
                    if best_swap and best_swap_score < class_distance:
                        idx, new_img, new_ann, old_was_overlap, new_is_overlap = best_swap
                        current_imgs[idx] = new_img
                        current_anns[idx] = new_ann
                        
                        # Update candidate pools
                        if new_is_overlap and (new_img, new_ann) in overlapping_candidates:
                            overlapping_candidates.remove((new_img, new_ann))
                        elif not new_is_overlap and (new_img, new_ann) in nonoverlapping_candidates:
                            nonoverlapping_candidates.remove((new_img, new_ann))
                    else:
                        break
                else:
                    break
            else:
                # Class overlap too high - would need similar swap logic
                break
    
    # Final measurements
    final_img_overlap = len(set(best_imgs) & base_img_set)
    final_class_overlap = measure_overlap(best_imgs, best_anns, base_classes, step_classes)
    
    print(f"    Final: {len(best_imgs)} images, {final_img_overlap} overlap with base ({final_img_overlap/len(best_imgs):.1%})")
    print(f"    Final class overlap: {final_class_overlap:.2%} (target: {target_class_overlap_ratio:.2%})")
    
    return best_imgs, best_anns

In [32]:
def run_dual_optimization_cooccurrence_based(
    initial_imgs=None, initial_anns=None,
    split_B_imgs=None, split_B_dict=None,
    base_imgs=None, base_anns=None,
    base_classes=None, step_classes=None,
    target_img_overlap_ratio=None, target_class_overlap_ratio=None,
    measure_overlap=None, threshold_k=5, max_iterations=100):
    """
    Dual optimization using co-occurrence ranking to achieve both targets.
    
    Strategy:
    - To DECREASE image overlap: Remove images with HIGHEST base co-occurrence,
      replace with split B images with LOWEST base co-occurrence
    - To INCREASE image overlap: Add images with HIGHEST base co-occurrence from split B
    - All replacements must belong to step (have step classes)
    
    Args:
        [same as before, plus:]
        threshold_k: Threshold for co-occurrence counting
        
    Returns:
        Tuple of (best_imgs, best_anns) achieving both targets
    """
    
    # Validate inputs
    if any(x is None for x in [initial_imgs, initial_anns, split_B_imgs, split_B_dict,
                                base_imgs, base_classes, step_classes, 
                                target_img_overlap_ratio, target_class_overlap_ratio,
                                measure_overlap]):
        raise ValueError("Missing required parameters")
    
    base_img_set = set(base_imgs)
    target_img_count = int(len(initial_imgs) * target_img_overlap_ratio)
    
    print(f"  Co-occurrence-Based Dual Optimization:")
    print(f"    Target image overlap: {target_img_count} / {len(initial_imgs)} ({target_img_overlap_ratio:.1%})")
    print(f"    Target class overlap: {target_class_overlap_ratio:.1%}")
    
    def compute_image_cooccurrence_score(img, ann, base_classes):
        """Compute co-occurrence score of an image with base classes."""
        img_classes = {seg['category_id'] for seg in ann['segments_info']}
        # Count how many base classes co-occur in this image
        return len(img_classes.intersection(base_classes))
    
    # Rank initial images by co-occurrence with base
    initial_ranked = []
    for img, ann in zip(initial_imgs, initial_anns):
        score = compute_image_cooccurrence_score(img, ann, base_classes)
        is_overlap = img in base_img_set
        initial_ranked.append((img, ann, score, is_overlap))
    
    # Sort by co-occurrence score (descending)
    initial_ranked.sort(key=lambda x: x[2], reverse=True)
    
    print(f"    Initial images ranked by base co-occurrence: {len(initial_ranked)}")
    
    # Build and rank split B candidates
    overlapping_candidates = []
    nonoverlapping_candidates = []
    
    for img in split_B_imgs:
        if img not in initial_imgs:
            ann = split_B_dict[img]
            img_classes = {seg['category_id'] for seg in ann['segments_info']}
            
            # Only consider images with step classes
            if img_classes & step_classes:
                score = compute_image_cooccurrence_score(img, ann, base_classes)
                
                if img in base_img_set:
                    overlapping_candidates.append((img, ann, score))
                else:
                    nonoverlapping_candidates.append((img, ann, score))
    
    # Sort candidates by co-occurrence score
    overlapping_candidates.sort(key=lambda x: x[2], reverse=True)  # Highest first
    nonoverlapping_candidates.sort(key=lambda x: x[2], reverse=True)  # Highest first
    
    overlapping_candidates = reorder_candidates_by_target(
        overlapping_candidates, target_class_overlap_ratio
    )
    nonoverlapping_candidates = reorder_candidates_by_target(
        nonoverlapping_candidates, target_class_overlap_ratio
    )
    
    print(f"    Split B candidates reordered for target class overlap {target_class_overlap_ratio:.1%}")
    print(f"    Available: {len(overlapping_candidates)} overlapping, "
          f"{len(nonoverlapping_candidates)} non-overlapping")
    
    # Current state
    current_overlap_imgs = [x for x in initial_ranked if x[3]]  # is_overlap=True
    current_nonoverlap_imgs = [x for x in initial_ranked if not x[3]]
    
    current_img_overlap_count = len(current_overlap_imgs)
    
    # Determine action needed
    if current_img_overlap_count > target_img_count:
        # DECREASE image overlap
        num_to_remove = current_img_overlap_count - target_img_count
        
        print(f"    Need to DECREASE overlap: remove {num_to_remove} overlapping images")
        
        # Remove images with HIGHEST base co-occurrence
        to_remove = current_overlap_imgs[:num_to_remove]  # Highest co-occurrence
        to_keep_overlap = current_overlap_imgs[num_to_remove:]
        
        # Replace with non-overlapping candidates with LOWEST base co-occurrence
        # that still achieve class overlap target
        replacement_pool = nonoverlapping_candidates #replacement_pool = nonoverlapping_candidates[::-1]  # Reverse to get lowest first
        
        # Iteratively select replacements that get closest to class overlap target
        replacements = []
        temp_imgs = [x[0] for x in to_keep_overlap] + [x[0] for x in current_nonoverlap_imgs]
        temp_anns = [x[1] for x in to_keep_overlap] + [x[1] for x in current_nonoverlap_imgs]
        
        cached_replacement_candidates = []
        
        for candidate_img, candidate_ann, candidate_score in replacement_pool:
            if len(replacements) >= num_to_remove:
                break
            
            # Test adding this replacement
            test_imgs = temp_imgs + [candidate_img]
            test_anns = temp_anns + [candidate_ann]
            test_class_overlap = measure_overlap(test_imgs, test_anns, base_classes, step_classes)
            prev_class_overlap = measure_overlap(temp_imgs, temp_anns, base_classes, step_classes)
            prev_distance = abs(prev_class_overlap - target_class_overlap_ratio)
            new_distance = abs(test_class_overlap - target_class_overlap_ratio)  # *** NEW ***

            
            
            # Accept if it moves us closer to target
            if new_distance <= prev_distance:
                replacements.append((candidate_img, candidate_ann, candidate_score))
                temp_imgs.append(candidate_img)
                temp_anns.append(candidate_ann)
                
                
            
            else: 
                # Would help preserve our image count
                cached_replacement_candidates.append({
                    'img': candidate_img,
                    'ann': candidate_ann,
                    'candidate_score': candidate_score,
                    'imp_dist': new_distance  # *** FIXED: Use new_distance ***
                })
        
        cached_replacement_candidates.sort(key=lambda x: x['imp_dist'])
        
        # Final images
        final_imgs = [x[0] for x in to_keep_overlap] + [x[0] for x in current_nonoverlap_imgs] + [x[0] for x in replacements]
        final_anns = [x[1] for x in to_keep_overlap] + [x[1] for x in current_nonoverlap_imgs] + [x[1] for x in replacements]
        
        diff = len(initial_imgs) - len(final_imgs)  # *** CHANGED: Calculate needed count ***
        
        if diff > 0:  # *** CHANGED: Need MORE images ***
            # Add from cache (already sorted by best improvement distance)
            to_add = min(diff, len(cached_replacement_candidates))  # *** NEW: Don't exceed available ***
            final_imgs += [cached_replacement_candidates[i]['img'] for i in range(to_add)]
            final_anns += [cached_replacement_candidates[i]['ann'] for i in range(to_add)]
        elif diff < 0:  # *** CHANGED: Have TOO MANY images (shouldn't happen but handle it) ***
            final_imgs = final_imgs[:len(initial_imgs)]
            final_anns = final_anns[:len(initial_imgs)]
        
        
        
        print(f"    Removed {len(to_remove)} high co-occurrence images")
        # print(f"    Added {len(replacements)} low co-occurrence replacements")
        
    elif current_img_overlap_count < target_img_count:
        # INCREASE image overlap
        num_to_add = target_img_count - current_img_overlap_count
        
        print(f"    Need to INCREASE overlap: add {num_to_add} overlapping images")
        
        # Add images with HIGHEST base co-occurrence from split B
        additions = []
        temp_imgs = [x[0] for x in initial_ranked]
        temp_anns = [x[1] for x in initial_ranked]
        
        cached_replacement_candidates = []
        
        for candidate_img, candidate_ann, candidate_score in overlapping_candidates:
            if len(additions) >= num_to_add:
                break
            
            # Test adding this candidate
            test_imgs = temp_imgs + [candidate_img]
            test_anns = temp_anns + [candidate_ann]
            test_class_overlap = measure_overlap(test_imgs, test_anns, base_classes, step_classes)
            prev_class_overlap = measure_overlap(temp_imgs, temp_anns, base_classes, step_classes)
            prev_distance = abs(prev_class_overlap - target_class_overlap_ratio)
            new_distance = abs(test_class_overlap - target_class_overlap_ratio)
            
            # Accept if it moves us closer to target
            if new_distance <= prev_distance:
            
                additions.append((candidate_img, candidate_ann, candidate_score))
                temp_imgs.append(candidate_img)
                temp_anns.append(candidate_ann)
            else:
                cached_replacement_candidates.append({
                    'img': candidate_img,
                    'ann': candidate_ann,
                    'candidate_score': candidate_score,
                    'imp_dist': new_distance  # *** NEW: Store distance ***
                })
                
        cached_replacement_candidates.sort(key=lambda x: x['imp_dist'])
        
        
        
        
        # # Final images
        # final_imgs = [x[0] for x in initial_ranked] + [x[0] for x in additions]
        # final_anns = [x[1] for x in initial_ranked] + [x[1] for x in additions]
        
        nonoverlap_ranked = sorted(current_nonoverlap_imgs, key=lambda x: x[2])
        num_to_remove = min(len(additions), len(nonoverlap_ranked))
        to_remove = nonoverlap_ranked[:num_to_remove]
        to_keep_nonoverlap = nonoverlap_ranked[num_to_remove:]
        
        # Final images
        final_imgs = ([x[0] for x in current_overlap_imgs] +
                      [x[0] for x in to_keep_nonoverlap] +
                      [x[0] for x in additions])
        
        final_anns = ([x[1] for x in current_overlap_imgs] +
                      [x[1] for x in to_keep_nonoverlap] +
                      [x[1] for x in additions])
        
        diff = len(initial_imgs) - len(final_imgs)
        
        if diff > 0:
            to_add = min(diff, len(cached_replacement_candidates))
            final_imgs += [cached_replacement_candidates[i]['img'] for i in range(to_add)]
            final_anns += [cached_replacement_candidates[i]['ann'] for i in range(to_add)]
        elif diff < 0:
            final_imgs = final_imgs[:len(initial_imgs)]
            final_anns = final_anns[:len(initial_imgs)]
        
        print(f"    Removed {len(to_remove)} low co-occurrence non-overlapping")
        print(f"    Added {len(additions)} target-optimized overlapping images")
        
    else:
        # Already at target image overlap, fine-tune class overlap only
        print(f"    Already at target image overlap, fine-tuning class overlap...")
        
        current_class_overlap = measure_overlap(
            [x[0] for x in initial_ranked],
            [x[1] for x in initial_ranked],
            base_classes, step_classes
        )
        
        if abs(current_class_overlap - target_class_overlap_ratio) < 0.01:
            # Already satisfactory
            final_imgs = [x[0] for x in initial_ranked]
            final_anns = [x[1] for x in initial_ranked]
        else:
            # Swap images to adjust class overlap
            final_imgs = [x[0] for x in initial_ranked]
            final_anns = [x[1] for x in initial_ranked]
            
            # Try swapping with candidates
            all_candidates = overlapping_candidates + nonoverlapping_candidates
            
            for i in range(min(10, len(final_imgs))):  # Try swapping up to 10 images
                best_swap = None
                best_swap_score = float('inf')
                
                for cand_img, cand_ann, cand_score in all_candidates[:20]:
                    # Simulate swap
                    test_imgs = final_imgs[:i] + [cand_img] + final_imgs[i+1:]
                    test_anns = final_anns[:i] + [cand_ann] + final_anns[i+1:]
                    test_class_overlap = measure_overlap(test_imgs, test_anns, base_classes, step_classes)
                    
                    score = abs(test_class_overlap - target_class_overlap_ratio)
                    if score < best_swap_score:
                        best_swap_score = score
                        best_swap = (i, cand_img, cand_ann)
                
                if best_swap and best_swap_score < abs(current_class_overlap - target_class_overlap_ratio):
                    idx, new_img, new_ann = best_swap
                    final_imgs[idx] = new_img
                    final_anns[idx] = new_ann
                    current_class_overlap = measure_overlap(final_imgs, final_anns, base_classes, step_classes)
    
    # Final verification
    final_img_overlap = len(set(final_imgs) & base_img_set)
    final_class_overlap = measure_overlap(final_imgs, final_anns, base_classes, step_classes)
    
    print(f"    Final: {len(final_imgs)} images, {final_img_overlap} overlap with base ({final_img_overlap/len(final_imgs):.1%})")
    print(f"    Final class overlap: {final_class_overlap:.2%} (target: {target_class_overlap_ratio:.2%})")
    print(f"    Image overlap achieved: {abs(final_img_overlap - target_img_count) <= 1}")
    print(f"    Class overlap achieved: {abs(final_class_overlap - target_class_overlap_ratio) < 0.05}")
    
    return final_imgs, final_anns



In [33]:
# def run_dual_optimization_cooccurrence_based2(
#     initial_imgs=None, initial_anns=None,
#     split_B_imgs=None, split_B_dict=None,
#     base_imgs=None, base_anns=None,
#     base_classes=None, step_classes=None,
#     target_img_overlap_ratio=None, target_class_overlap_ratio=None,
#     measure_overlap=None, threshold_k=5, max_iterations=100):
#     """
#     Dual optimization using co-occurrence ranking to achieve both targets.
    
#     Strategy:
#     - To DECREASE image overlap: Remove images with HIGHEST base co-occurrence,
#       replace with split B images with LOWEST base co-occurrence
#     - To INCREASE image overlap: Add images with HIGHEST base co-occurrence from split B
#     - All replacements must belong to step (have step classes)
    
#     Args:
#         [same as before, plus:]
#         threshold_k: Threshold for co-occurrence counting
        
#     Returns:
#         Tuple of (best_imgs, best_anns) achieving both targets
#     """
    
#     # Validate inputs
#     if any(x is None for x in [initial_imgs, initial_anns, split_B_imgs, split_B_dict,
#                                 base_imgs, base_classes, step_classes, 
#                                 target_img_overlap_ratio, target_class_overlap_ratio,
#                                 measure_overlap]):
#         raise ValueError("Missing required parameters")
    
#     base_img_set = set(base_imgs)
#     target_img_count = int(len(initial_imgs) * target_img_overlap_ratio)
    
#     print(f"  Co-occurrence-Based Dual Optimization:")
#     print(f"    Target image overlap: {target_img_count} / {len(initial_imgs)} ({target_img_overlap_ratio:.1%})")
#     print(f"    Target class overlap: {target_class_overlap_ratio:.1%}")
    
#     def compute_image_cooccurrence_score(img, ann, base_classes):
#         """Compute co-occurrence score of an image with base classes."""
#         img_classes = {seg['category_id'] for seg in ann['segments_info']}
#         # Count how many base classes co-occur in this image
#         # return measure_overlap([img], [ann], base_classes, step_classes, threshold_k=1)
#         return len(img_classes.intersection(base_classes))
    
#     # Rank initial images by co-occurrence with base
#     initial_ranked = []
#     for img, ann in zip(initial_imgs, initial_anns):
#         score = compute_image_cooccurrence_score(img, ann, base_classes)
#         is_overlap = img in base_img_set
#         initial_ranked.append((img, ann, score, is_overlap))
    
#     # Sort by co-occurrence score (descending)
#     initial_ranked.sort(key=lambda x: x[2], reverse=True)
    
#     print(f"    Initial images ranked by base co-occurrence: {len(initial_ranked)}")
    
#     # Build and rank split B candidates
#     overlapping_candidates = []
#     nonoverlapping_candidates = []
    
#     for img in split_B_imgs:
#         if img not in initial_imgs:
#             ann = split_B_dict[img]
#             img_classes = {seg['category_id'] for seg in ann['segments_info']}
            
#             # Only consider images with step classes
#             if img_classes & step_classes:
#                 score = compute_image_cooccurrence_score(img, ann, base_classes)
                
#                 if img in base_img_set:
#                     overlapping_candidates.append((img, ann, score))
#                 else:
#                     nonoverlapping_candidates.append((img, ann, score))
    
#     # Sort candidates by co-occurrence score
#     overlapping_candidates.sort(key=lambda x: x[2], reverse=True)  # Highest first
#     nonoverlapping_candidates.sort(key=lambda x: x[2], reverse=True)  # Highest first
    
#     print(f"    Split B candidates: {len(overlapping_candidates)} overlapping, "
#           f"{len(nonoverlapping_candidates)} non-overlapping")
    
#     # Current state
#     current_overlap_imgs = [x for x in initial_ranked if x[3]]  # is_overlap=True
#     current_nonoverlap_imgs = [x for x in initial_ranked if not x[3]]
    
#     current_img_overlap_count = len(current_overlap_imgs)
    
    
#     # Determine action needed
#     if current_img_overlap_count > target_img_count:
#         # DECREASE image overlap
#         num_to_remove = current_img_overlap_count - target_img_count
        
#         print(f"    Need to DECREASE overlap: remove {num_to_remove} overlapping images")
        
#         # Remove images with HIGHEST base co-occurrence
#         to_remove = current_overlap_imgs[:num_to_remove]  # Highest co-occurrence
#         to_keep_overlap = current_overlap_imgs[num_to_remove:]
        
#         # Replace with non-overlapping candidates with LOWEST base co-occurrence
#         # that still achieve class overlap target
#         replacement_pool = nonoverlapping_candidates[::-1]  # Reverse to get lowest first
        
#         # Iteratively select replacements that get closest to class overlap target
#         replacements = []
#         temp_imgs = [x[0] for x in to_keep_overlap] + [x[0] for x in current_nonoverlap_imgs]
#         temp_anns = [x[1] for x in to_keep_overlap] + [x[1] for x in current_nonoverlap_imgs]
#         cached_replacement_candidates = []
        
        
        
#         for candidate_img, candidate_ann, candidate_score in replacement_pool:
#             if len(replacements) >= num_to_remove:
#                 break
            
#             # Test adding this replacement
#             test_imgs = temp_imgs + [candidate_img]
#             test_anns = temp_anns + [candidate_ann]
#             test_class_overlap = measure_overlap(test_imgs, test_anns, base_classes, step_classes)
#             prev_class_overlap = measure_overlap(temp_imgs, temp_anns, base_classes, step_classes)
            
#             # *** FIXED: Correct improvement calculation ***
#             # Accept if distance to target DECREASES
#             prev_distance = abs(prev_class_overlap - target_class_overlap_ratio)
#             new_distance = abs(test_class_overlap - target_class_overlap_ratio)
#             improvement = new_distance <= prev_distance  # *** NEW: Correct comparison ***
            
#             if improvement:
#                 replacements.append((candidate_img, candidate_ann, candidate_score))
#                 temp_imgs.append(candidate_img)
#                 temp_anns.append(candidate_ann)
#             else: 
#                 # Would help preserve our image count
#                 cached_replacement_candidates.append({
#                     'img': candidate_img,
#                     'ann': candidate_ann,
#                     'candidate_score': candidate_score,
#                     'imp_dist': new_distance  # *** FIXED: Use new_distance ***
#                 })
        
#         # *** NEW: Sort cache by improvement distance (best first) ***
#         cached_replacement_candidates.sort(key=lambda x: x['imp_dist'])
        
#         # Final images
#         final_imgs = [x[0] for x in to_keep_overlap] + [x[0] for x in current_nonoverlap_imgs] + [x[0] for x in replacements]
#         final_anns = [x[1] for x in to_keep_overlap] + [x[1] for x in current_nonoverlap_imgs] + [x[1] for x in replacements]
        
#         # *** FIXED: Ensure final count matches initial ***
#         diff = len(initial_imgs) - len(final_imgs)  # *** CHANGED: Calculate needed count ***
        
#         if diff > 0:  # *** CHANGED: Need MORE images ***
#             # Add from cache (already sorted by best improvement distance)
#             to_add = min(diff, len(cached_replacement_candidates))  # *** NEW: Don't exceed available ***
#             final_imgs += [cached_replacement_candidates[i]['img'] for i in range(to_add)]
#             final_anns += [cached_replacement_candidates[i]['ann'] for i in range(to_add)]
#         elif diff < 0:  # *** CHANGED: Have TOO MANY images (shouldn't happen but handle it) ***
#             final_imgs = final_imgs[:len(initial_imgs)]
#             final_anns = final_anns[:len(initial_imgs)]
#         print(f"    Cached was: {len(cached_replacement_candidates)}, replacements: {len(replacements)}")
#         print(f"    Removed {len(to_remove)} high co-occurrence overlapping images")
#         print(f"    Added {len(replacements)} optimal + {max(0, diff - len(replacements))} cached replacements")
#         print(f"    Final count: {len(final_imgs)} (target: {len(initial_imgs)})")

#     elif current_img_overlap_count < target_img_count:
#         # INCREASE image overlap
#         num_to_add = target_img_count - current_img_overlap_count
        
#         print(f"    Need to INCREASE overlap: add {num_to_add} overlapping images")
        
#         # Add images with HIGHEST base co-occurrence from split B
#         additions = []
#         temp_imgs = [x[0] for x in initial_ranked]
#         temp_anns = [x[1] for x in initial_ranked]
#         cached_replacement_candidates = []
        
        
#         for candidate_img, candidate_ann, candidate_score in overlapping_candidates:
#             if len(additions) >= num_to_add:
#                 break
            
#             # Test adding this candidate
#             test_imgs = temp_imgs + [candidate_img]
#             test_anns = temp_anns + [candidate_ann]
#             test_class_overlap = measure_overlap(test_imgs, test_anns, base_classes, step_classes)
#             prev_class_overlap = measure_overlap(temp_imgs, temp_anns, base_classes, step_classes)
            
#             # *** FIXED: Correct improvement calculation ***
#             prev_distance = abs(prev_class_overlap - target_class_overlap_ratio)
#             new_distance = abs(test_class_overlap - target_class_overlap_ratio)
#             improvement = new_distance <= prev_distance   # *** NEW: Correct comparison ***
            
#             if improvement:
#                 additions.append((candidate_img, candidate_ann, candidate_score))
#                 temp_imgs.append(candidate_img)
#                 temp_anns.append(candidate_ann)
#             else: 
#                 # Would help preserve our image overlap
#                 cached_replacement_candidates.append({
#                     'img': candidate_img,
#                     'ann': candidate_ann,
#                     'candidate_score': candidate_score,
#                     'imp_dist': new_distance  # *** FIXED: Use new_distance ***
#                 })
        
#         # *** NEW: Sort cache by improvement distance (best first) ***
#         cached_replacement_candidates.sort(key=lambda x: x['imp_dist'])
        
#         # *** NEW: Remove non-overlapping images to make room ***
#         nonoverlap_ranked = sorted(current_nonoverlap_imgs, key=lambda x: x[2])
#         num_to_remove = min(len(additions), len(nonoverlap_ranked))  # *** NEW: Don't remove more than available ***
#         to_remove = nonoverlap_ranked[:num_to_remove]
#         to_keep_nonoverlap = nonoverlap_ranked[num_to_remove:]
        
#         # Final images
#         final_imgs = ([x[0] for x in current_overlap_imgs] +
#                     [x[0] for x in to_keep_nonoverlap] +
#                     [x[0] for x in additions])
        
#         final_anns = ([x[1] for x in current_overlap_imgs] +
#                     [x[1] for x in to_keep_nonoverlap] +
#                     [x[1] for x in additions])
        
#         # *** FIXED: Ensure final count matches initial ***
#         diff = len(initial_imgs) - len(final_imgs)  # *** CHANGED: Calculate needed count ***
        
#         if diff > 0:  # *** CHANGED: Need MORE images ***
#             # Add from cache (already sorted by best improvement distance)
#             to_add = min(diff, len(cached_replacement_candidates))  # *** NEW: Don't exceed available ***
#             final_imgs += [cached_replacement_candidates[i]['img'] for i in range(to_add)]
#             final_anns += [cached_replacement_candidates[i]['ann'] for i in range(to_add)]
#         elif diff < 0:  # *** CHANGED: Have TOO MANY images ***
#             final_imgs = final_imgs[:len(initial_imgs)]
#             final_anns = final_anns[:len(initial_imgs)]
#         print(f"    Cached was: {len(cached_replacement_candidates)}, additions: {len(additions)}")
#         print(f"    Removed {len(to_remove)} low co-occurrence non-overlapping images")
#         print(f"    Added {len(additions)} optimal + {max(0, diff - len(additions))} cached overlapping images")
#         print(f"    Final count: {len(final_imgs)} (target: {len(initial_imgs)})")

#     else:
#         # Already at target image overlap, fine-tune class overlap only
#         print(f"    Already at target image overlap, fine-tuning class overlap...")
        
#         current_class_overlap = measure_overlap(
#             [x[0] for x in initial_ranked],
#             [x[1] for x in initial_ranked],
#             base_classes, step_classes
#         )
        
#         if abs(current_class_overlap - target_class_overlap_ratio) < 0.01:
#             # Already satisfactory
#             final_imgs = [x[0] for x in initial_ranked]
#             final_anns = [x[1] for x in initial_ranked]
#             print(f"    Class overlap already at target ({current_class_overlap:.2%})")
#         else:
#             # Swap images to adjust class overlap
#             final_imgs = [x[0] for x in initial_ranked]
#             final_anns = [x[1] for x in initial_ranked]
#             img_set = set(final_imgs)
            
#             # Candidates (not already in current set, and matching the overlap category)
#             all_candidates = []
#             for cand_img, cand_ann, cand_score in overlapping_candidates + nonoverlapping_candidates:
#                 if cand_img not in img_set:
#                     all_candidates.append((cand_img, cand_ann, cand_score))
            
#             swaps_made = 0  # *** NEW: Track swaps ***
#             for i in range(min(20, len(final_imgs))):  # Try swapping up to 20 images
#                 orig_img = final_imgs[i]
#                 orig_ann = final_anns[i]
#                 orig_overlap = orig_img in base_img_set
#                 best_swap = None
#                 best_swap_score = float('inf')
                
#                 for cand_img, cand_ann, cand_score in all_candidates:
#                     cand_overlap = cand_img in base_img_set
#                     # Only allow swap if overlap status remains the same (preserve image overlap count exactly)
#                     if cand_overlap == orig_overlap:
#                         # Simulate swap
#                         test_imgs = final_imgs[:i] + [cand_img] + final_imgs[i+1:]
#                         test_anns = final_anns[:i] + [cand_ann] + final_anns[i+1:]
#                         test_class_overlap = measure_overlap(test_imgs, test_anns, base_classes, step_classes)
                        
#                         score = abs(test_class_overlap - target_class_overlap_ratio)
#                         if score < best_swap_score:
#                             best_swap_score = score
#                             best_swap = (i, cand_img, cand_ann)
                
#                 if best_swap and best_swap_score < abs(current_class_overlap - target_class_overlap_ratio):
#                     idx, new_img, new_ann = best_swap
#                     final_imgs[idx] = new_img
#                     final_anns[idx] = new_ann
#                     img_set.remove(orig_img)
#                     img_set.add(new_img)
#                     current_class_overlap = measure_overlap(final_imgs, final_anns, base_classes, step_classes)
#                     swaps_made += 1  # *** NEW: Count swap ***
                    
#                     # Remove used candidate from future swaps to avoid duplicates
#                     all_candidates = [c for c in all_candidates if c[0] != new_img]
            
#             print(f"    Performed {swaps_made} swaps to adjust class overlap")  # *** NEW: Report swaps ***
#             print(f"    Final class overlap: {current_class_overlap:.2%}")
    
#     # Final verification
#     final_img_overlap = len(set(final_imgs) & base_img_set)
#     final_class_overlap = measure_overlap(final_imgs, final_anns, base_classes, step_classes)
    
#     print(f"    Final: {len(final_imgs)} images, {final_img_overlap} overlap with base ({final_img_overlap/len(final_imgs):.1%})")
#     print(f"    Final class overlap: {final_class_overlap:.2%} (target: {target_class_overlap_ratio:.2%})")
#     print(f"    Image overlap achieved: {abs(final_img_overlap - target_img_count) <= 1}")
#     print(f"    Class overlap achieved: {abs(final_class_overlap - target_class_overlap_ratio) < 0.05}")
    
#     return final_imgs, final_anns
 
 

In [34]:

def adjust_image_overlap(initial_step_data, split_B_imgs, split_B_anns, 
                         base_cls, inc_cls, target_img_overlap_ratio,
                         measure_class_overlap, target_class_overlap_ratio=None, num_classes=150,
                         approach='image_overlap_only',max_iterations=100):
    """
    Adjusts image overlap between incremental steps and base by swapping images with split B.
    
    Args:
        initial_step_data: dict from create_initial_setup: {step: (imgs, anns)}
        split_B_imgs: List of image IDs from split B (reserve pool)
        split_B_anns: List of annotations from split B
        base_cls: Number of base classes
        inc_cls: Number of incremental classes per step
        target_img_overlap_ratio: Target overlap ratio (0.0 to 1.0)
        measure_class_overlap: Function to measure class overlap
        num_classes: Total number of classes
        
    Returns:
        pd.DataFrame: New setup table with adjusted overlaps
        dict: {step: (imgs, anns)} with adjusted image sets
    """
    class_order = list(range(num_classes))
    base_classes = set(class_order[:base_cls])
    
    # Get base step (unchanged)
    base_imgs, base_anns = initial_step_data[1]
    base_img_set = set(base_imgs)
    
    # Build split B lookup
    split_B_dict = {img: ann for img, ann in zip(split_B_imgs, split_B_anns)}
    
    new_step_data = {1: (base_imgs, base_anns)}  # Base unchanged
    table_rows = []
    
    # Add base to table
    base_avail_classes = set()
    for ann in base_anns:
        base_avail_classes.update(seg['category_id'] for seg in ann['segments_info'])
    
    table_rows.append({
        'step': 1,
        'num_imgs': len(base_imgs),
        'imgs_ov_base': len(base_imgs),
        'img_ov_%': round(len(base_imgs)/len(base_imgs),2),
        'avail_cls': len(base_avail_classes),
        'ov_cls_base': 0.0,
        'step_cls': len(base_classes)
    })
    
    print(f"\nAdjusting image overlap to target: {target_img_overlap_ratio:.1%}")
    print("="*60)
    
    # Process each incremental step
    for step in range(2, len(initial_step_data) + 1):
        start_idx = base_cls + (step - 2) * inc_cls
        end_idx = min(start_idx + inc_cls, num_classes)
        step_classes = set(class_order[start_idx:end_idx])
        
        # Get initial step images
        initial_imgs, initial_anns = initial_step_data[step]
        initial_dict = {img: ann for img, ann in zip(initial_imgs, initial_anns)}
        
        initial_overlap_ratio, initial_overlap_count = compute_image_overlap_with_base(
            initial_imgs, base_imgs
        )
        
        target_count = int(len(initial_imgs) * target_img_overlap_ratio)
        
        
        if approach =='image_overlap_only' or approach == 'class_overlap_on_image_overlap' or approach is None:
            print(f"\nStep {step}:")
            print(f"  Initial: {len(initial_imgs)} images, {initial_overlap_count} overlap "
                f"({initial_overlap_ratio:.1%})")
            print(f"  Target: {target_count} overlap images ({target_img_overlap_ratio:.1%})")
            
            # Separate overlapping and non-overlapping images
            overlap_imgs = [img for img in initial_imgs if img in base_img_set]
            non_overlap_imgs = [img for img in initial_imgs if img not in base_img_set]
            
            # Build annotation lookup for initial step
            
        
            if target_count > len(overlap_imgs):
                # Need to add overlapping images from split B
                needed = target_count - len(overlap_imgs)
                candidates = []
                
                for img in split_B_imgs: # this is where I might want to control for class overlap too
                    if img in base_img_set and img not in initial_imgs:
                        ann = split_B_dict[img]
                        img_classes = {seg['category_id'] for seg in ann['segments_info']}
                        if img_classes & step_classes:  # Has step classes
                            candidates.append((img, ann))
                
                 # Add needed images
                if approach == 'class_overlap_on_image_overlap':
                    added = select_images_dual_objective_iterative(
                        candidates, 
                        overlap_imgs, 
                        [initial_dict[img] for img in overlap_imgs],
                        base_imgs, base_anns, base_classes, step_classes,
                        needed, measure_overlap, target_class_overlap_ratio, base_img_set,
                        mode='add',max_iterations=max_iterations
                    )
                else:
                    added = candidates[:needed]
                
                final_imgs = overlap_imgs + [img for img, _ in added] + non_overlap_imgs
                final_anns = ([initial_dict[img] for img in overlap_imgs] + 
                            [ann for _, ann in added] +
                            [initial_dict[img] for img in non_overlap_imgs])
                
                print(f"  Added {len(added)} overlapping images from split B")
                
            elif target_count < len(overlap_imgs):
                # Need to remove overlapping images, replace with non-overlapping from split B
                keep_overlap = overlap_imgs[:target_count]
                remove_overlap = overlap_imgs[target_count:]
                
                # Find replacements from split B (non-overlapping with base)
                candidates = []
                for img in split_B_imgs:
                    if img not in base_img_set and img not in initial_imgs:
                        ann = split_B_dict[img]
                        img_classes = {seg['category_id'] for seg in ann['segments_info']}
                        if img_classes & step_classes:
                            candidates.append((img, ann))
                
                # Replace removed images
                if approach == 'class_overlap_on_image_overlap':
                    replacements = select_images_dual_objective_iterative(
                        candidates,
                        keep_overlap + non_overlap_imgs,
                        [initial_dict[img] for img in keep_overlap + non_overlap_imgs],
                        base_imgs, base_anns, base_classes, step_classes,
                        len(remove_overlap), measure_overlap, target_class_overlap_ratio, base_img_set,
                        # Adding non-overlapping replacements
                        mode='add',max_iterations=max_iterations
                    )
                else:
                    replacements = candidates[:len(remove_overlap)]
                    
                final_imgs = (keep_overlap + non_overlap_imgs + 
                            [img for img, _ in replacements])
                final_anns = ([initial_dict[img] for img in keep_overlap] +
                            [initial_dict[img] for img in non_overlap_imgs] +
                            [ann for _, ann in replacements])
                
                print(f"  Removed {len(remove_overlap)} overlapping images")
                print(f"  Added {len(replacements)} non-overlapping images from split B")
            else:
                # Already at target
                final_imgs = initial_imgs
                final_anns = initial_anns
                print(f"  Already at target overlap")

        elif approach == 'dual_optimization':
            
            final_imgs, final_anns = run_dual_optimization(
                initial_imgs=initial_imgs,  initial_anns=initial_anns,
                split_B_imgs=split_B_imgs, split_B_dict=split_B_dict,
                base_imgs=base_imgs, base_anns=base_anns,
                base_classes=base_classes, step_classes=step_classes,
                target_img_overlap_ratio=target_img_overlap_ratio, 
                target_class_overlap_ratio=target_class_overlap_ratio,
                measure_overlap=measure_overlap, max_iterations=max_iterations
            )
        elif approach == 'dual_optimization_cooccurrence':
            final_imgs, final_anns = run_dual_optimization_cooccurrence_based(
                initial_imgs=initial_imgs, initial_anns=initial_anns,
                split_B_imgs=split_B_imgs, split_B_dict=split_B_dict,
                base_imgs=base_imgs, base_anns=base_anns,
                base_classes=base_classes, step_classes=step_classes,
                target_img_overlap_ratio=target_img_overlap_ratio, 
                target_class_overlap_ratio=target_class_overlap_ratio,
                measure_overlap=measure_class_overlap,
                max_iterations=max_iterations
            )
            # final_imgs, final_anns = run_dual_optimization_cooccurrence_based(
            #     initial_imgs=final_imgs, initial_anns=final_anns,
            #     split_B_imgs=split_B_imgs, split_B_dict=split_B_dict,
            #     base_imgs=base_imgs, base_anns=base_anns,
            #     base_classes=base_classes, step_classes=step_classes,
            #     target_img_overlap_ratio=target_img_overlap_ratio, 
            #     target_class_overlap_ratio=target_class_overlap_ratio,
            #     measure_overlap=measure_class_overlap,
            #     max_iterations=max_iterations
            # )
        
        final_imgs=  list(set(final_imgs)) 
              
        # Verify and measure
        final_overlap_ratio, final_overlap_count = compute_image_overlap_with_base(
            final_imgs, base_imgs
        )
        
        step_avail_classes = set()
        for ann in final_anns:
            step_avail_classes.update(seg['category_id'] for seg in ann['segments_info'])
        
        overlap_score = measure_class_overlap(final_imgs, final_anns, base_classes, step_classes)
        
        new_step_data[step] = (final_imgs, final_anns)
        
        table_rows.append({
            'step': step,
            'num_imgs': len(final_imgs),
            'imgs_ov_base': final_overlap_count,
            'img_ov_%': round(final_overlap_count/len(final_imgs),2),
            'avail_cls': len(step_avail_classes),
            'ov_cls_base': overlap_score,
            'step_cls': len(step_classes)
        })
        
        print(f"  Final: {len(final_imgs)} images, {final_overlap_count} overlap "
              f"({final_overlap_ratio:.1%})")
        

    incremental_rows = [row for row in table_rows if row['step'] != 1]
    
    tt_row_num_imgs = sum(row['num_imgs'] for row in table_rows)
    tt_row_imgs_ov_base = np.mean([row['imgs_ov_base'] for row in incremental_rows]) if incremental_rows else 0
    tt_row_img_ov = np.mean([row['img_ov_%'] for row in incremental_rows]) if incremental_rows else 0
    
    table_rows.append({
        'step': 'Total',
        'num_imgs': tt_row_num_imgs,
        'imgs_ov_base': f"μ: {tt_row_imgs_ov_base:.2f}",
        'img_ov_%': f"μ: {tt_row_img_ov:.2f}",
        'avail_cls': f"μ: {np.mean([row['avail_cls'] for row in incremental_rows]):.2f}",
        'ov_cls_base': f"μ: {np.mean([row['ov_cls_base'] for row in incremental_rows]) if incremental_rows else 0:.2f}",
        'step_cls': '-'
    })
    
    df = pd.DataFrame(table_rows)
    print("\n" + "="*60)
    print("Adjustment complete!")
    return df, new_step_data



In [35]:
def plot_overlap_table(df, img_overlap_pct, cls_overlap_pct, scenario="100-50",
                      output_path="overlap_table.png", title_prefix="",
                      subtitle_info=None):
    """
    Plot formatted table for overlap analysis results.
    
    Args:
        df: DataFrame with columns ['step', 'num_imgs', 'imgs_ov_base', 'img_ov_%', 
                                    'avail_cls', 'ov_cls_base', 'step_cls']
        img_overlap_pct: Image overlap percentage for title (e.g., 50)
        cls_overlap_pct: Class overlap percentage for title (e.g., 75)
        scenario: Scenario string (e.g., "100-50")
        output_path: Path to save the figure
        title_prefix: Optional prefix for title (e.g., "Split A")
        subtitle_info: Dict with optional subtitle details (e.g., {'seed': 42, 'order': 'random'})
    """
    
    # Create figure
    num_rows = len(df)
    fig, ax = plt.subplots(figsize=(15, max(8, num_rows * 0.6 + 3)))
    ax.axis('off')
    
    # Build headers
    header_row1 = ['Step', 'Images', 'Ov Imgs', 'Img Ov %', 'Avail Cls', 'Cls Ov Base', 'Step Cls']
    
    # Build table data from DataFrame
    table_data = []
    for _, row in df.iterrows():
        table_row = [
            str(row['step']),
            str(row['num_imgs']),
            str(row['imgs_ov_base']),
            f"{row['img_ov_%']:.2f}" if isinstance(row['img_ov_%'], (int, float)) else str(row['img_ov_%']),
            str(row['avail_cls']),
            f"{row['ov_cls_base']:.2f}" if isinstance(row['ov_cls_base'], (int, float)) else str(row['ov_cls_base']),
            str(row['step_cls'])
        ]
        table_data.append(table_row)
    
    # Combine headers and data
    all_table_data = [header_row1] + table_data
    
    # Create table
    table = ax.table(cellText=all_table_data,
                     cellLoc='center',
                     loc='center',
                     bbox=[0, 0, 1, 1])
    
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 2.5)
    
    # Color header row
    num_cols = len(header_row1)
    for j in range(num_cols):
        if j == 0:
            table[(0, j)].set_facecolor('#4472C4')
        else:
            table[(0, j)].set_facecolor('#70AD47')
        table[(0, j)].set_text_props(weight='bold', color='white')
    
    # Color data rows
    num_rows = len(all_table_data)
    for i in range(1, num_rows):
        for j in range(num_cols):
            if j == 0:  # Step column
                table[(i, j)].set_facecolor('#D9E1F2')
                table[(i, j)].set_text_props(weight='bold')
            elif i == num_rows - 1:  # Total row
                table[(i, j)].set_facecolor('#FFF2CC')
                table[(i, j)].set_text_props(weight='bold')
            else:
                # Alternate colors for different columns
                if j in [1]:  # num_imgs
                    table[(i, j)].set_facecolor('#E2EFDA')
                elif j in [2, 3]:  # imgs_ov_base, img_ov_%
                    table[(i, j)].set_facecolor('#FFF2CC')
                elif j in [4]:  # avail_cls
                    table[(i, j)].set_facecolor('#FFE699')
                elif j in [5]:  # ov_cls_base
                    table[(i, j)].set_facecolor('#FCE4D6')
                else:  # step_cls
                    table[(i, j)].set_facecolor('#E2EFDA')
    
    # Title and subtitle
    cls_overlap_pct = cls_overlap_pct if cls_overlap_pct == "Default" else cls_overlap_pct
    title = f"Target Image Overlap: {img_overlap_pct}% | Target Class Overlap: {cls_overlap_pct}% | Scenario: {scenario}"
    
    subtitle = ""
    if subtitle_info:
        if 'seed' in subtitle_info:
            subtitle += f"Seed: {subtitle_info['seed']}"
        if 'order' in subtitle_info:
            subtitle += f" | Class Order: 0 - 149" # {subtitle_info['order']}"
        if 'method' in subtitle_info:
            subtitle += f" | Method: {subtitle_info['method']}"
    
    plt.suptitle(title, fontsize=16, fontweight='bold', y=0.95)
        
    if subtitle:
        plt.title(subtitle, fontsize=15, fontweight='bold', y=1)
    
    # Save
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"✅ Saved table: {output_path}")





In [36]:
def format_float_columns(df, decimals=2):
    """Format all float columns to specified decimal places."""
    float_cols = df.select_dtypes(include=['float']).columns
    df[float_cols] = df[float_cols].round(decimals)
    return df



In [37]:
initial_df, initial_step_data = create_initial_setup(
    all_splitA_imgs, all_splitA_ann, 
    base_cls=100, inc_cls=50, 
    measure_overlap=measure_overlap
)


initial_df = format_float_columns(initial_df, decimals=2)

Base step: 10177 images with classes [0, 1, 2, 3, 4]...
Step 2: 4658 images, 4655 overlap with base, class overlap: 93.00%


In [38]:

print("\nInitial Setup:")
print(initial_df)

base_cls, inc_cls = 100, 50

scenario=f"{base_cls}-{inc_cls}"

plot_overlap_table(
    initial_df, 
    img_overlap_pct='Default', 
    cls_overlap_pct='Default',
    scenario=scenario,
    output_path=f"analyses/{scenario}/default_cls_ov/overlap_initial_table_100-50.png",
    title_prefix="Split A",
    subtitle_info={'seed': 42, 'order': 'random', 'method': 'image overlap'}
)


Initial Setup:
    step  num_imgs imgs_ov_base img_ov_%  avail_cls ov_cls_base step_cls
0      1     10177        10177      1.0        150         0.0      100
1      2      4658         4655      1.0        150        0.93       50
2  Total     14835   μ: 4655.00  μ: 1.00  μ: 150.00     μ: 0.93        -
✅ Saved table: analyses/100-50/default_cls_ov/overlap_initial_table_100-50.png


In [39]:
#approaches: class_overlap_on_image_overlap, dual_optimization, dual_optimization_cooccurrence
for img_ov in [0.75, 0.5, 0.25, 0]:  
    for cls_ov in ["Default"]:# 0.5, 0.25, 0]: 
        approach = 'image_overlap_only'
        adjusted_df_with_ov, adjusted_data = adjust_image_overlap(
            initial_step_data, all_splitB_imgs, all_splitB_ann,
            base_cls=100, inc_cls=50, target_img_overlap_ratio=img_ov,
            measure_class_overlap=measure_overlap,target_class_overlap_ratio=cls_ov,
            approach=approach,max_iterations=5
        )

        scenario=f"{base_cls}-{inc_cls}"

        adjusted_df_with_ov = format_float_columns(adjusted_df_with_ov, decimals=2)
        
        print(f"\nAdjusted Setup ({img_ov} image overlap):")
        print(adjusted_df_with_ov)
        
        plot_overlap_table(
            adjusted_df_with_ov, 
            img_overlap_pct=f"{img_ov*100}", 
            cls_overlap_pct=f"{cls_ov}",
            scenario=scenario,
            output_path=f"analyses/{scenario}/default_cls_ov/overlap_{int(img_ov*100)}_{cls_ov}_100-50.png",
            title_prefix="Split A",
            subtitle_info={'seed': 42, 'order': 'random', 'method': approach}
        )



Adjusting image overlap to target: 75.0%

Step 2:
  Initial: 4658 images, 4655 overlap (99.9%)
  Target: 3493 overlap images (75.0%)
  Removed 1162 overlapping images
  Added 1162 non-overlapping images from split B
  Final: 4658 images, 3493 overlap (75.0%)

Adjustment complete!

Adjusted Setup (0.75 image overlap):
    step  num_imgs imgs_ov_base img_ov_%  avail_cls ov_cls_base step_cls
0      1     10177        10177      1.0        150         0.0      100
1      2      4658         3493     0.75        150        0.89       50
2  Total     14835   μ: 3493.00  μ: 0.75  μ: 150.00     μ: 0.89        -
✅ Saved table: analyses/100-50/default_cls_ov/overlap_75_Default_100-50.png

Adjusting image overlap to target: 50.0%

Step 2:
  Initial: 4658 images, 4655 overlap (99.9%)
  Target: 2329 overlap images (50.0%)
  Removed 2326 overlapping images
  Added 2326 non-overlapping images from split B
  Final: 4658 images, 2329 overlap (50.0%)

Adjustment complete!

Adjusted Setup (0.5 image ove

In [40]:

def save_overlap_split(df: pd.DataFrame, 
                       step_data: Dict[int, Tuple[List, List]], 
                       base_output_dir: str,
                       scenario: str = "100-50",
                       additional_metadata: Dict = None) -> str:
    """
    Save dataframe and step data to directory named by mean overlaps.
    
    Args:
        df: DataFrame with overlap statistics
        step_data: Dict mapping step -> (img_ids, annotations)
        base_output_dir: Base directory for saving
        scenario: Scenario identifier (e.g., "100-50")
        additional_metadata: Optional dict with extra info to save
        
    Returns:
        Path to saved directory
    """
    
    # Extract mean overlaps from the Total row
    total_row = df[df['step'] == 'Total'].iloc[0]
    
    # Parse mean values from strings like "μ: 0.75"
    mean_img_ov_str = total_row['img_ov_%']
    mean_cls_ov_str = total_row['ov_cls_base']
    
    if isinstance(mean_img_ov_str, str):
        mean_img_ov = float(mean_img_ov_str.split(': ')[1])
    else:
        mean_img_ov = float(mean_img_ov_str)
    
    if isinstance(mean_cls_ov_str, str):
        mean_cls_ov = float(mean_cls_ov_str.split(': ')[1])
    else:
        mean_cls_ov = float(mean_cls_ov_str)
    
    # Create directory name
    dir_name = f"img_ov{int(mean_img_ov * 100)}_cls_ov{int(mean_cls_ov * 100)}"
    save_dir = Path(base_output_dir) / scenario / dir_name
    save_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"Saving to: {save_dir}")
    
    # Save DataFrame as CSV
    df_path = save_dir / "overlap_stats.csv"
    df.to_csv(df_path, index=False)
    print(f"  ✓ Saved DataFrame: {df_path}")
    
    # Save step data (image IDs only for efficiency)
    step_data_dict = {}
    for step, (imgs, anns) in step_data.items():
        # Extract image IDs (assuming imgs are either IDs or dicts with 'id')
        if imgs and isinstance(imgs[0], dict):
            img_ids = [img['id'] if 'id' in img else img.get('file_name', str(i)) 
                      for i, img in enumerate(imgs)]
        else:
            img_ids = imgs  # Already IDs
        
        step_data_dict[int(step)] = {
            'image_ids': img_ids,
            'num_images': len(img_ids)
        }
    
    step_data_path = save_dir / "step_image_ids.json"
    with open(step_data_path, 'w') as f:
        json.dump(step_data_dict, f, indent=2)
    print(f"  ✓ Saved step image IDs: {step_data_path}")
    
    # Save metadata
    metadata = {
        'scenario': scenario,
        'mean_img_overlap': float(mean_img_ov),
        'mean_cls_overlap': float(mean_cls_ov),
        'total_images': int(total_row['num_imgs']) if total_row['num_imgs'] != 'Total' else 0,
        'num_steps': len(step_data),
        'created_at': pd.Timestamp.now().isoformat()
    }
    
    if additional_metadata:
        metadata.update(additional_metadata)
    
    metadata_path = save_dir / "metadata.json"
    with open(metadata_path, 'w') as f:
        json.dump(metadata, f, indent=2)
    print(f"  ✓ Saved metadata: {metadata_path}")
    
    print(f"✅ Successfully saved split to: {save_dir}")
    return str(save_dir)

 

In [41]:
## approaches: class_overlap_on_image_overlap, dual_optimization, dual_optimization_cooccurrence
# for cls_ov in [0.85, 0.75, 0.5, 0.25, 0]:  
#     for img_ov in [0.75, 0.5, 0.25, 1, 0]: 

 
for img_ov in [1,.75,.5,.25,0]: 
    for cls_ov in [1, .5,  0]: #[1,.75, .5, .25, 0]: 
      
        base_cls, inc_cls = 100, 50

        scenario=f"{base_cls}-{inc_cls}"
        
        approach = 'dual_optimization_cooccurrence' #'dual_optimization'
        adjusted_df_with_ov, adjusted_data = adjust_image_overlap(
            initial_step_data, all_splitB_imgs, all_splitB_ann,
            base_cls=base_cls, inc_cls=inc_cls, target_img_overlap_ratio=img_ov,
            measure_class_overlap=measure_overlap,target_class_overlap_ratio=cls_ov,
            approach=approach,max_iterations=300
        )


        adjusted_df_with_ov = format_float_columns(adjusted_df_with_ov, decimals=2)
        
        print(f"\nAdjusted Setup ({img_ov} image overlap):")
        print(adjusted_df_with_ov)
        
        save_path = save_overlap_split(
            df=adjusted_df_with_ov,
            step_data=adjusted_data,
            base_output_dir=f"analyses/{scenario}/custom_high_mid_low_cls_ov",
            scenario=scenario,
            additional_metadata={
                'method': approach,
                'target_img_ov': img_ov,
                'target_cls_ov': cls_ov
            }
        )

        
        plot_overlap_table(
            adjusted_df_with_ov, 
            img_overlap_pct=f"{img_ov*100}", 
            cls_overlap_pct=f"{cls_ov*100}",
            scenario=scenario,
            output_path=f"analyses/{scenario}/custom_high_mid_low_cls_ov/overlap_{int(img_ov*100)}_{int(cls_ov*100)}_100-50.png",
            title_prefix="Split A",
            subtitle_info={'seed': 42, 'order': '-', 'method': approach}
        )





Adjusting image overlap to target: 100.0%
  Co-occurrence-Based Dual Optimization:
    Target image overlap: 4658 / 4658 (100.0%)
    Target class overlap: 100.0%
    Initial images ranked by base co-occurrence: 4658
    Split B candidates reordered for target class overlap 100.0%
    Available: 0 overlapping, 4732 non-overlapping
    Need to INCREASE overlap: add 3 overlapping images
    Removed 0 low co-occurrence non-overlapping
    Added 0 target-optimized overlapping images
    Final: 4658 images, 4655 overlap with base (99.9%)
    Final class overlap: 93.00% (target: 100.00%)
    Image overlap achieved: False
    Class overlap achieved: False
  Final: 4658 images, 4655 overlap (99.9%)

Adjustment complete!

Adjusted Setup (1 image overlap):
    step  num_imgs imgs_ov_base img_ov_%  avail_cls ov_cls_base step_cls
0      1     10177        10177      1.0        150         0.0      100
1      2      4658         4655      1.0        150        0.93       50
2  Total     14835   μ: